# Regularizing: smaller, L2, and dropout

Three ways to fight overfitting on the same problem, measured against each other rather than described.

**Runs on:** CPU — about 4 minutes &nbsp;·&nbsp; **Slides:** [Chapter 5 — Fundamentals of Machine Learning](../../../course-web-slides/ch05/index.html) &nbsp;·&nbsp; **Section:** 05 — Improving generalization

---

## The setup, and the baseline to beat

In [ ]:
import numpy as np
import keras
from keras import layers, regularizers
from keras.datasets import imdb

(train_data, train_labels), _ = imdb.load_data(num_words=10000)

def vectorize(seqs, dim=10000):
    out = np.zeros((len(seqs), dim), dtype="float32")
    for i, s in enumerate(seqs):
        out[i, s] = 1.
    return out

x = vectorize(train_data)
y = np.asarray(train_labels).astype("float32")

def fit(model, epochs=20):
    model.compile(optimizer="rmsprop", loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model.fit(x, y, epochs=epochs, batch_size=512,
                     validation_split=0.4, verbose=0)

keras.utils.set_random_seed(0)
baseline = fit(keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
]))

## Option 1: a smaller model

In [ ]:
keras.utils.set_random_seed(0)
smaller = fit(keras.Sequential([
    layers.Dense(4, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
]))

## Option 2: L2 weight regularization

Add the sum of the squared weights to the loss. The model now pays for complexity, so it keeps only the weights that earn their place.

In [ ]:
keras.utils.set_random_seed(0)
l2 = fit(keras.Sequential([
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.002),
                 activation="relu"),
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.002),
                 activation="relu"),
    layers.Dense(1, activation="sigmoid"),
]))

## Option 3: dropout

Zero a random half of the outputs during training. The layer cannot rely on any specific unit being present, so it cannot build a fragile conspiracy of units that happens to fit the training set.

In [ ]:
keras.utils.set_random_seed(0)
dropout = fit(keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
]))

> **Note** — Dropout is active during **training only**. At inference Keras scales the outputs instead, so `predict()` is deterministic — which is why you never see it in the evaluation numbers.

## All four, on one axis

In [ ]:
import matplotlib.pyplot as plt

runs = [("baseline (16, 16)", baseline, "#888888"),
        ("smaller (4, 4)", smaller, "#1f77b4"),
        ("L2 0.002", l2, "#2ca02c"),
        ("dropout 0.5", dropout, "#d62728")]

plt.figure(figsize=(8, 4.8))
for name, h, c in runs:
    plt.plot(h.history["val_loss"], lw=1.7, c=c, label=name)
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.ylim(0.25, 0.75)
plt.legend(); plt.title("Three ways to delay overfitting")
plt.show()

print(f"{'run':22s} {'best val loss':>14s} {'at epoch':>9s}")
for name, h, _ in runs:
    v = h.history["val_loss"]
    print(f"{name:22s} {min(v):14.4f} {int(np.argmin(v))+1:9d}")

Read the **epoch of the minimum**, not just the minimum. Regularization does not usually give a dramatically better best score on this problem — it delays the turn, which buys you a wider window in which the model is good.

## Dropout rate is a real hyperparameter

In [ ]:
rates = [0.0, 0.2, 0.5, 0.8]
best = []
for r in rates:
    keras.utils.set_random_seed(0)
    layers_list = [layers.Dense(16, activation="relu")]
    if r: layers_list.append(layers.Dropout(r))
    layers_list.append(layers.Dense(16, activation="relu"))
    if r: layers_list.append(layers.Dropout(r))
    layers_list.append(layers.Dense(1, activation="sigmoid"))
    h = fit(keras.Sequential(layers_list), epochs=15)
    best.append(min(h.history["val_loss"]))
    print(f"dropout {r:.1f} -> best val loss {best[-1]:.4f}")

plt.figure(figsize=(5.5, 3.8))
plt.plot(rates, best, "o-")
plt.xlabel("dropout rate"); plt.ylabel("best validation loss")
plt.title("Too much dropout underfits")
plt.show()

At 0.8 the model is being asked to work with a fifth of its units and it underfits. **Regularization taken far enough becomes the other failure** — which is why chapter 18 hands the choice to a tuner rather than to intuition.

## What actually works best

The most effective regularizer on this problem is not on the chart: **more training data**. Notebook 02 showed the curve. Every technique here is what you reach for when more data is not available — which is most of the time, and is why they matter.

---

## What to take away

- Reducing capacity, L2, and dropout all delay overfitting by different mechanisms.
- Read the epoch at which validation loss turns, not only its minimum.
- Dropout is training-only; inference is deterministic.
- Over-regularizing produces underfitting — the rate is a hyperparameter, not a constant.